# Connect to Genie API

| Step | What you'll do |
|------|---------------|
| **Step 1** | Connect to the Genie API and ask a question |
| **Step 2** | Wrap it in a simple orchestrator (1 Genie) |
| **Step 3** | Scale to multiple Genie Spaces |

In [0]:
%pip install -qU databricks-langchain

dbutils.library.restartPython()

## Imports

In [0]:
import json
import time
from datetime import timedelta
from typing import Any  # for type hints

import mlflow
import pandas as pd
from databricks.sdk import WorkspaceClient  # Connection to Databricks SDK + authentication
from databricks.sdk.service.dashboards import GenieMessage  # Response type from Genie
from databricks_langchain import ChatDatabricks
from IPython.display import Markdown, display

## Configure Variables

* genie_space_id: str | Genie Space -> Configure -> Settings -> Space ID

In [0]:
dbutils.widgets.text("genie_space_id", "01f0fe3570de1a69a78b1b9b71d392ff", "1. Genie Space ID")

### Save to global variables

In [0]:
GENIE_SPACE_ID = dbutils.widgets.get("genie_space_id")

print(GENIE_SPACE_ID)

## Step 1: Connect to Genie through Workspace SDK

* Use the [WorkspaceClient library](https://databricks-sdk-py.readthedocs.io/en/latest/workspace/dashboards/genie.html$0) with for U2M authentication
* Create a conversation and parse the response with `print_output(message)`

In [0]:
# Create a WorkspaceClient for authentication
w = ...  # TODO

In [0]:
# Use w.genie.start_conversation_and_wait() to ask Genie a question
# Params: space_id, content (your question), timeout
msg = ...  # TODO
print(msg)

### GenieMessage Structure

`start_conversation_and_wait` returns a `GenieMessage` with these key fields:

| Field | Type | Description |
|-------|------|-------------|
| `content` | `str` | The original question you sent |
| `conversation_id` | `str` | Reuse this for follow-up questions |
| `message_id` | `str` | Unique ID for this message |
| `attachments` | `List[GenieAttachment]` | AI-generated response — SQL, text answer, and/or suggested follow-ups |
| `status` | `MessageStatus` | `COMPLETED`, `FAILED`, etc. |
| `error` | `MessageError \| None` | Error details if status is `FAILED` |

**Attachments** are the interesting part — each one can contain:
- `.query` — the generated SQL and its description
- `.text` — the natural-language answer
- `.suggested_questions` — follow-up question suggestions

Not every attachment has all three; check for `None` before accessing.

### Parse Message

In [0]:
# Extract the text answer from msg.attachments
answer = next((att.text.content for att in msg.attachments if att.text and att.text.content), None)
print(answer)

## Helper Functions

| Method | Purpose |
|--------|---------|
| `print_output(message)` | Format a `GenieMessage` into a readable dict (question, SQL, answer) |
| `ask_genie(query, space_id)` | Send a question to a Genie Space and return a parsed result dict |
| `get_space_metadata(space_id)` | Fetch table schemas, sample questions, and descriptions for a space |
| `to_dataframe(result)` | Convert a Genie result into a `pandas.DataFrame` via `get_message_attachment_query_result` |
| `ask_genie_verbose(query, space_id)` | Like `ask_genie` but prints each status transition during polling |

In [ ]:
class GenieUtils:
    @staticmethod
    def print_output(message: GenieMessage) -> None:
        """Pretty-print a GenieMessage: question, SQL, and answer."""
        sql = None
        answer = None
        for att in message.attachments or []:
            if att.query and att.query.query and sql is None:
                sql = att.query.query
            if att.text and att.text.content and answer is None:
                answer = att.text.content

        print(f"Question:\n  {message.content}\n")
        if sql:
            print(f"SQL:\n  {sql}\n")
        if answer:
            print(f"Answer:\n  {answer}")

    @staticmethod
    def ask_genie(query: str, space_id: str, client: WorkspaceClient = None) -> dict:
        if client is None:
            raise ValueError("client (WorkspaceClient) is required")

        msg = client.genie.start_conversation_and_wait(
            space_id=space_id,
            content=query,
            timeout=timedelta(seconds=120),
        )

        msg_dict = msg.as_dict() if hasattr(msg, "as_dict") else vars(msg)

        sql, description, text = None, None, None
        text_contents = []

        for att in msg_dict.get("attachments") or []:
            if att.get("query"):
                sql = sql or att["query"].get("query")
                description = description or att["query"].get("description")
            if att.get("text"):
                content = att["text"].get("content")
                if content:
                    text_contents.append(content)

        text = max(text_contents, key=len) if text_contents else None

        return {"message": msg, "sql": sql, "description": description, "data": None, "text": text}

    @staticmethod
    def to_dataframe(result: dict, client: WorkspaceClient = None) -> "pd.DataFrame":
        """Convert a Genie result dict into a pandas DataFrame.

        Uses get_message_attachment_query_result to fetch the tabular data
        from the first query attachment. Falls back to a single-column
        DataFrame containing the text answer if no query attachment exists.
        """
        if client is None:
            raise ValueError("client (WorkspaceClient) is required")

        msg = result["message"]
        msg_dict = msg.as_dict() if hasattr(msg, "as_dict") else vars(msg)

        # Find the first query attachment with a statement_id
        for att in msg_dict.get("attachments") or []:
            query_att = att.get("query")
            if query_att and query_att.get("statement_id"):
                qr = client.genie.get_message_attachment_query_result(
                    space_id=msg.space_id,
                    conversation_id=msg.conversation_id,
                    message_id=msg.id,
                    attachment_id=att["attachment_id"],
                )
                # qr is a GenieGetMessageQueryResultResponse
                # Access: qr.statement_response.manifest.schema.columns / .result.data_array
                stmt = qr.statement_response
                columns = [c.name for c in (stmt.manifest.schema.columns or [])]
                rows = stmt.result.data_array or []
                return pd.DataFrame(rows, columns=columns)

        # Fallback: return the text answer as a single-row DataFrame
        return pd.DataFrame([{"answer": result.get("text", "")}])

    @staticmethod
    def ask_genie_verbose(query: str, space_id: str, client: WorkspaceClient = None) -> dict:
        """Query Genie with verbose status output.

        Uses the lower-level start_conversation + polling loop so you can
        see each status transition (ASKING_AI -> EXECUTING_QUERY -> COMPLETED).
        Returns the same dict format as ask_genie.
        """
        if client is None:
            raise ValueError("client (WorkspaceClient) is required")

        # Start conversation (non-blocking)
        conv = client.genie.start_conversation(
            space_id=space_id,
            content=query,
        )
        conversation_id = conv.conversation_id
        message_id = conv.message_id
        print(f"[verbose] Conversation started: {conversation_id}")
        print(f"[verbose] Message ID: {message_id}")

        last_status = None
        while True:
            msg = client.genie.get_message(
                space_id=space_id,
                conversation_id=conversation_id,
                message_id=message_id,
            )
            current_status = msg.status.value if msg.status else "UNKNOWN"
            if current_status != last_status:
                print(f"[verbose] Status: {current_status}")
                last_status = current_status

            if current_status in ("COMPLETED", "FAILED"):
                break
            time.sleep(2)

        if current_status == "FAILED":
            error = msg.error if hasattr(msg, "error") else "Unknown error"
            print(f"[verbose] Genie returned an error: {error}")
            return {"message": msg, "sql": None, "description": None, "data": None, "text": None}

        # Parse the completed message (same logic as ask_genie)
        msg_dict = msg.as_dict() if hasattr(msg, "as_dict") else vars(msg)
        sql, description, text = None, None, None
        text_contents = []

        for att in msg_dict.get("attachments") or []:
            if att.get("query"):
                sql = sql or att["query"].get("query")
                description = description or att["query"].get("description")
            if att.get("text"):
                content = att["text"].get("content")
                if content:
                    text_contents.append(content)

        text = max(text_contents, key=len) if text_contents else None
        print(f"[verbose] Done — SQL: {'yes' if sql else 'no'}, Text: {'yes' if text else 'no'}")

        return {"message": msg, "sql": sql, "description": description, "data": None, "text": text}

    @staticmethod
    def get_space_metadata(space_id: str, client: WorkspaceClient = None, enrich_columns: bool = False) -> dict:
        """Fetch metadata for a Genie Space.

        Args:
            enrich_columns: When True, fetches full column schemas from
                Unity Catalog for every table in the space. When False,
                only returns manually-annotated column_configs from the
                Genie Space configuration.
        """
        if client is None:
            raise ValueError("client (WorkspaceClient) is required")

        space = client.genie.get_space(
            space_id=space_id,
            include_serialized_space=True,
        )

        serialized = space.serialized_space
        if not serialized:
            raise RuntimeError("serialized_space missing in response (check permissions: need CAN EDIT on the space)")

        cfg = json.loads(serialized)
        config = cfg.get("config", {})
        data_sources = cfg.get("data_sources", {})

        sample_questions = [" ".join(q.get("question", [])) for q in config.get("sample_questions", [])]

        tables = []
        for t in data_sources.get("tables", []):
            identifier = t.get("identifier")
            # Genie Space column annotations (manually configured)
            genie_columns = [
                {
                    "column_name": col_meta.get("column_name"),
                    "description": col_meta.get("description"),
                }
                for col_meta in t.get("column_configs", [])
            ]

            # Optionally enrich with Unity Catalog schema
            uc_columns = []
            if enrich_columns and identifier:
                try:
                    uc_table = client.tables.get(full_name=identifier)
                    uc_columns = [
                        {
                            "column_name": col.name,
                            "type": str(col.type_name.value) if col.type_name else None,
                            "comment": col.comment,
                        }
                        for col in (uc_table.columns or [])
                    ]
                except Exception as e:
                    uc_columns = [{"error": str(e)}]

            table_info = {
                "identifier": identifier,
                "description": t.get("description"),
                "columns": uc_columns if enrich_columns else genie_columns,
            }
            tables.append(table_info)

        return {
            "space_id": space.space_id,
            "title": space.title,
            "description": space.description,
            "sample_questions": sample_questions,
            "tables": tables,
        }

In [0]:
GenieUtils.print_output(msg)

### Multi-Turn Conversations

Every Genie response includes a `conversation_id`. Passing it back with `create_message_and_wait` lets you ask follow-up questions in the same conversation — Genie remembers the SQL context, so "break that down by region" works without restating the original question.

In [0]:
# Get the conversation ID from the first message
conversation_id = msg.conversation_id
print(f"Conversation ID: {conversation_id}")

In [0]:
# Follow up in the same conversation — Genie remembers context
# Use w.genie.create_message_and_wait() with the conversation_id from above
followup = ...  # TODO
GenieUtils.print_output(followup)

### Status Transitions

When Genie processes a question it moves through a lifecycle:

| Status | Meaning |
|--------|---------|
| `SUBMITTED` | Request received |
| `ASKING_AI` | LLM is generating SQL |
| `EXECUTING_QUERY` | SQL is running on the warehouse |
| `COMPLETED` | Results are ready |

`ask_genie_verbose` uses the low-level `start_conversation` + `get_message` polling loop so you can watch each transition in real time.

In [0]:
# Verbose mode: see each status transition as Genie processes your question
GenieUtils.ask_genie_verbose("Which salesperson has the highest total sales this year?", GENIE_SPACE_ID, client=w)

### Pandas Display

`GenieUtils.to_dataframe(result)` calls `get_message_attachment_query_result` under the hood to fetch the raw rows from the query attachment, then wraps them in a `pandas.DataFrame` for easy display and downstream analysis.

In [0]:
# 1. Use GenieUtils.ask_genie() to query Genie
# 2. Use GenieUtils.to_dataframe() to convert the result to a DataFrame
result = ...  # TODO
df = ...  # TODO
df

## Connect to LLM
* Use ChatDatabricks to connect to an endpoint - for a list view AI/ML -> Serving in the tab on the left hand side
* Bonus - Set up MLFlow autologging

In [0]:
mlflow.set_tracking_uri("databricks")
mlflow.set_registry_uri("databricks-uc")
mlflow.langchain.autolog()

In [0]:
# Create a ChatDatabricks LLM instance
# Hint: endpoint="databricks-gpt-5-nano", temperature=1
llm = ...  # TODO

In [0]:
# Test the LLM with a simple question
llm.invoke(...)  # TODO

## Step 2: Orchestrator (Single Genie)

Now that we can query Genie and parse its response, let's wrap everything into an **orchestrator** — a thin layer that:
1. Reads space metadata to understand available tables
2. Uses an LLM to rewrite and route the user's question
3. Sends the improved query to the right Genie Space
4. Formats the result for the user

In [0]:
# Use GenieUtils.ask_genie() to ask: "How many orders were placed last month by payment method?"
# TODO


### Space Metadata for Routing

`get_space_metadata` returns the table schemas, column descriptions, and sample questions configured in a Genie Space. The router LLM needs this context to decide **which space** can answer a question and to **rewrite** the query so Genie produces the best SQL.

In [0]:
# Fetch space metadata with enrich_columns=True to see full Unity Catalog schemas
# TODO


### LLM Router

`ask_router` takes the user's question plus a list of Genie Space IDs. For each space it fetches metadata (tables, columns, sample questions), then asks the LLM to:
1. **Rewrite** the question to match the available schema
2. **Pick** the best space to answer it

The output is a dict with `{"query": "...", "space_id": "..."}`.

In [0]:
def ask_router(
    llm: ChatDatabricks,
    question: str,
    genie_spaces: list[str],
    client: WorkspaceClient = None,
) -> dict[str, Any]:
    prompt = f"""
You are an intelligent router and orchestrator for Databricks Genie AI, an agent proficient in natural language -> SQL generation and retrieval. You are given a question from a user, which may be incomplete or inaccurate relative to the data available, and your task is to improve and augment the question to be maximally useful to Genie.

Use best practices and your knowledge of the available tables to improve the question to elicit the most effective response from Genie possible. 

Return only a dictionary in the format below:
{{
    "query": "The improved version of the question. Type: string",
    "space_id": "The genie space to which to route the question. Type: string"
}}

Question: {question}
"""
    # TODO: Append space metadata to the prompt so the LLM knows what data is available
    # Loop over genie_spaces, call GenieUtils.get_space_metadata() for each, and append to prompt
    prompt += "\n\nGenie Spaces Available and their metadata\n"
    for space_id in genie_spaces:
        pass  # fill in

    # TODO: Call the LLM with the prompt and parse the JSON response
    response = ...  # fill in
    obj = ...  # fill in (hint: json.loads on the response content)
    if "query" not in obj or "space_id" not in obj:
        raise ValueError("Invalid response from LLM")

    return obj

In [0]:
# Call ask_router with the LLM, a question, and a list of space IDs
router_response = ...  # TODO

print(router_response)

### GenieOrchestrator Class

`GenieOrchestrator` wires the three pieces together:
- **Router** — rewrites the question and picks the right Genie Space
- **Genie** — executes the query via `ask_genie`
- **LLM formatter** — produces a human-friendly summary of the result

Call `orchestrator.execute(question)` to run the full pipeline.

In [0]:
class GenieOrchestrator:
    def __init__(
        self,
        llm,
        space_ids: list[str] = None,
        client: WorkspaceClient = None,
    ):
        self.llm = llm
        self.space_ids = list(space_ids) if space_ids else [GENIE_SPACE_ID]
        self.client = client

        # Cache space titles at init time (1 lightweight API call per space)
        self.space_titles = {}
        for sid in self.space_ids:
            self._cache_title(sid)

    def _cache_title(self, space_id: str):
        """Fetch and cache the title for a single space."""
        try:
            space = self.client.genie.get_space(space_id=space_id)
            self.space_titles[space_id] = space.title
        except Exception:
            self.space_titles[space_id] = space_id

    def execute(self, user_question: str, output_format: str = "full"):
        """Run the full orchestration pipeline.

        Args:
            output_format: How to display results.
                "full" — print question, SQL, and answer (default, no LLM cost)
                "text" — print only the text answer
                "llm"  — use the LLM to produce a polished summary
                "raw"  — no printing, just return the dict
        """
        # TODO: Call ask_router() to get the rewritten query and target space_id
        router_response = ...  # fill in

        routed_id = router_response["space_id"]
        routed_title = self.space_titles.get(routed_id, routed_id)
        print(f"Routed to space: {routed_title} ({routed_id})")
        print(f"Rewritten query: {router_response['query']}\n")

        # TODO: Call GenieUtils.ask_genie() with the router's rewritten query and chosen space
        genie_response = ...  # fill in

        if output_format == "full":
            print(f"Question:\n  {user_question}\n")
            if genie_response.get("sql"):
                print(f"SQL:\n  {genie_response['sql']}\n")
            if genie_response.get("text"):
                print(f"Answer:\n  {genie_response['text']}")
        elif output_format == "text":
            print(genie_response.get("text", "No text answer returned."))
        elif output_format == "llm":
            summary = self.llm.invoke(
                f"Format the following Genie response into a clean, readable summary for the user:\n{genie_response}"
            )
            display(Markdown(summary.content))
        # "raw" — no printing

        return genie_response

    def add_space(self, space_id: str):
        self.space_ids.append(space_id)
        self._cache_title(space_id)

In [0]:
# Create a GenieOrchestrator with the LLM and workspace client
orch = ...  # TODO

In [0]:
# Run the orchestrator with a question and output_format="llm"
# TODO


## Multi Genie Orchestration

Real organisations split data across multiple Genie Spaces (Sales, Customers, Inventory, etc.).

An orchestrator queries **all spaces** and combines the results.

```
Question
   |
   +---> Sales Genie      ---> revenue data
   +---> Customers Genie   ---> segment data
   +---> Inventory Genie   ---> stock data
   |
   v
Combined Results
```

In [0]:
resp = w.genie.list_spaces()
for space in resp.spaces:
    if space.title.startswith("Velocity"):
        print(space.space_id, space.title)

In [0]:
# Domain-specific Genie Space IDs (deployed by 00c_setup_genie)
# Replace these with your actual space IDs after running setup
DOMAIN_SPACES = [
    "01f0fe356cbe15a7a4259f6822d1ebe8",  # Velocity Motors - Sales Analytics
    "01f0fe356c771e348b333b15f82c2e15",  # Velocity Motors - Customer Intelligence
    "01f0fe356c511a278a49fdc1e83797e3",  # Velocity Motors - Operations & Inventory
]

multi_orch = GenieOrchestrator(
    llm=llm,
    space_ids=DOMAIN_SPACES,
    client=w,
)

print(f"Multi-Genie orchestrator initialized with {len(multi_orch.space_ids)} spaces")

### Router in Action

The router now reads metadata from **all 3 spaces** before deciding where to send each question. Watch the `Routed to space:` output — different questions should land on different spaces.

In [0]:
# Sales domain — should route to Sales Analytics space
multi_orch.execute("What are the top 5 selling vehicle models by total revenue?")

In [0]:
# CRM domain — should route to Customer Intelligence space
multi_orch.execute("Which customer segments generate the most service revenue?")

In [0]:
# Operations domain — should route to Operations & Inventory space
multi_orch.execute("Which parts are below their reorder point?")

### Cross-Domain Questions

The real power of multi-space routing: ask an ambiguous question that could touch multiple domains. The router picks the best space based on table metadata.

In [0]:
# Cross-domain — router must decide: is this CRM, Sales, or Operations?
multi_orch.execute("What is the customer lifetime value including service history?")

## Cross-Domain Synthesis

The router picks **one** space per question. But some questions span multiple domains:

> _"Compare revenue by region with customer satisfaction ratings and parts inventory levels"_

No single Genie Space has all the data. The solution: **decompose → fan-out → synthesize**.

```
Complex Question
       |
       v
   Decomposer (LLM)
       |
       +---> Sub-query 1 → Sales Genie      → revenue by region
       +---> Sub-query 2 → CRM Genie        → satisfaction ratings
       +---> Sub-query 3 → Operations Genie  → inventory levels
       |
       v
   Synthesizer (LLM)
       |
       v
   Unified Answer
```

In [0]:
def ask_decomposer(
    llm: ChatDatabricks,
    question: str,
    genie_spaces: list[str],
    client: WorkspaceClient = None,
) -> list[dict[str, str]]:
    """Break a complex question into sub-queries, each targeted at a specific Genie Space.

    Returns a list of {"query": "...", "space_id": "..."} dicts.
    """
    prompt = f"""You are a query decomposer for Databricks Genie AI. You are given a complex question
that may require data from multiple Genie Spaces (each space covers a different data domain).

Your job is to break the question into focused sub-queries, each targeting exactly one Genie Space.
Each sub-query should be self-contained and answerable by a single space.

Return ONLY a JSON list of objects in this format:
[
    {{"query": "The focused sub-question for this space", "space_id": "The target space ID"}}
]

Rules:
- Each sub-query must target exactly one space_id from the list below
- Keep sub-queries simple and specific — Genie works best with focused questions
- Use 1-4 sub-queries (only as many as needed)
- If the question only needs one space, return a single-element list

Question: {question}
"""
    # TODO: Append space metadata (same pattern as ask_router)
    prompt += "\n\nGenie Spaces Available and their metadata:\n"
    for space_id in genie_spaces:
        pass  # fill in

    # TODO: Call the LLM and parse the JSON list of sub-queries
    response = ...  # fill in
    sub_queries = ...  # fill in

    if not isinstance(sub_queries, list):
        raise ValueError(f"Expected a list from decomposer, got: {type(sub_queries)}")

    return sub_queries

In [0]:
def execute_multi(self, user_question: str, output_format: str = "full"):
    """Decompose a complex question across multiple spaces and synthesize results.

    1. Decomposer LLM breaks the question into targeted sub-queries
    2. Each sub-query runs against its assigned Genie Space
    3. Synthesizer LLM combines all results into a unified answer

    Args:
        output_format: "full", "text", "llm", or "raw" (same as execute)
    """
    # Step 1: Decompose
    print(f"Decomposing: {user_question}\n")
    # TODO: Call ask_decomposer() to break the question into sub-queries
    sub_queries = ...  # fill in

    print(f"Plan: {len(sub_queries)} sub-queries")
    for i, sq in enumerate(sub_queries, 1):
        title = self.space_titles.get(sq["space_id"], sq["space_id"])
        print(f"  {i}. [{title}] {sq['query']}")
    print()

    # Step 2: Fan-out — execute each sub-query
    results = []
    for i, sq in enumerate(sub_queries, 1):
        title = self.space_titles.get(sq["space_id"], sq["space_id"])
        print(f"Running sub-query {i}/{len(sub_queries)}: {title}...")

        try:
            # TODO: Call GenieUtils.ask_genie() for this sub-query
            response = ...  # fill in
            results.append(
                {
                    "space_id": sq["space_id"],
                    "space_title": title,
                    "query": sq["query"],
                    "sql": response.get("sql"),
                    "text": response.get("text"),
                    "success": True,
                }
            )
            print("  Done.\n")
        except Exception as e:
            results.append(
                {
                    "space_id": sq["space_id"],
                    "space_title": title,
                    "query": sq["query"],
                    "sql": None,
                    "text": None,
                    "success": False,
                    "error": str(e),
                }
            )
            print(f"  Failed: {e}\n")

    # Step 3: Synthesize
    synthesis_context = "\n\n".join(
        f"--- {r['space_title']} ---\nQuery: {r['query']}\nAnswer: {r.get('text', 'No answer')}" for r in results
    )

    synthesis_prompt = f"""You are a data analyst synthesizer. You were given a complex question that was
broken into sub-queries across different data domains. Below are the results from each domain.

Original question: {user_question}

Sub-query results:
{synthesis_context}

Produce a unified, well-structured answer that combines insights from all domains.
Be specific with numbers and highlight cross-domain patterns."""

    # TODO: Call the LLM with the synthesis prompt to combine all results
    synthesis = ...  # fill in

    if output_format == "full":
        print(f"Question:\n  {user_question}\n")
        for r in results:
            print(f"[{r['space_title']}]")
            if r.get("sql"):
                print(f"  SQL: {r['sql']}")
            if r.get("text"):
                print(f"  Result: {r['text']}")
            print()
        print(f"Synthesized Answer:\n  {synthesis.content}")
    elif output_format == "text":
        print(synthesis.content)
    elif output_format == "llm":
        display(Markdown(synthesis.content))
    # "raw" — no printing

    return {
        "question": user_question,
        "sub_queries": sub_queries,
        "results": results,
        "synthesis": synthesis.content,
    }


# Patch onto GenieOrchestrator
GenieOrchestrator.execute_multi = execute_multi

### Try It

These questions intentionally span multiple domains — the decomposer should create sub-queries for 2–3 different spaces.

In [0]:
# Spans Sales + CRM + Operations
multi_orch.execute_multi(
    "Compare revenue by region with customer satisfaction ratings and parts inventory levels",
    output_format="llm",
)

In [0]:
# Spans Sales + Operations
multi_orch.execute_multi(
    "Which vehicle models have the highest service costs relative to their sale price?",
    output_format="llm",
)

---
## Summary

| What you did | Code |
|---|---|
| Connect to Genie | `WorkspaceClient()` + `genie.start_conversation_and_wait()` |
| Parse response | Loop over `message.attachments` |
| Get data rows | `GenieUtils.to_dataframe(result, client)` → `pandas.DataFrame` |
| Follow-up | `genie.create_message_and_wait()` with `conversation_id` |
| Verbose polling | `GenieUtils.ask_genie_verbose()` — watch status transitions |
| Space metadata | `GenieUtils.get_space_metadata()` — table schemas + sample questions |
| LLM routing | `ask_router()` — rewrite question + pick target space |
| Single-space orchestration | `GenieOrchestrator.execute()` — router → Genie → format |
| Multi-space routing | Pass multiple space IDs → router picks the best one |
| Cross-domain synthesis | `GenieOrchestrator.execute_multi()` — decompose → fan-out → synthesize |

**The entire Genie API boils down to 3 SDK calls:**
1. `start_conversation_and_wait` — new question
2. `create_message_and_wait` — follow-up
3. `get_message_attachment_query_result` — get the data